# Consolidado anual de velas Binance

Descomprime los zips que dejó `descarga_binance.ipynb` y concatena, por año, los CSV mensuales **tal como vienen**: se copian los bytes, sin parsear ni transformar. Mismo formato que Binance: sin encabezado, 12 columnas.

- Si un mes no tiene archivo mensual, se usan sus diarios (el mes en curso).
- Se descarta una línea de encabezado sólo si algún archivo la trae, para que el consolidado no la tenga en medio.
- Ojo al leerlo: desde 2025 `open_time` y `close_time` vienen en **microsegundos**; antes, en milisegundos.

## 1. Imports

In [1]:
import gzip
import re
import zipfile
from pathlib import Path

## 1.1 [CONFIG]

In [2]:
RUTA         = Path(r"C:\Users\56jua\Desktop\git_tesis\bayesian_non_parametrics-1\data\reales\raw\cripto_binance")
PAR          = "BTCUSDT"
INTERVALO    = "5m"
ANIOS        = None     # None = todos los encontrados; o p. ej. [2023, 2024]
COMPRIMIR    = False    # True -> .csv.gz (unas 5x menos peso, más lento de escribir y leer)
SOBRESCRIBIR = False

## 2. Inventario de zips

In [3]:
CACHE  = RUTA / PAR / INTERVALO / "zip"
SALIDA = RUTA / PAR / INTERVALO / "anual"
patron = re.compile(rf"{PAR}-{INTERVALO}-(\d{{4}})-(\d{{2}})(?:-(\d{{2}}))?\.zip$")

mensuales, diarios = {}, {}
for zp in CACHE.rglob("*.zip"):
    m = patron.search(zp.name)
    if not m:
        continue
    clave = (int(m[1]), int(m[2]))
    if m[3] is None:
        mensuales[clave] = zp
    else:
        diarios.setdefault(clave, []).append(zp)

# por año, zips en orden temporal; el mensual manda sobre los diarios del mismo mes
por_anio = {}
for clave in sorted(set(mensuales) | set(diarios)):
    archivos = [mensuales[clave]] if clave in mensuales else sorted(diarios[clave])
    por_anio.setdefault(clave[0], []).extend(archivos)

if ANIOS is not None:
    por_anio = {a: v for a, v in por_anio.items() if a in ANIOS}
assert por_anio, f"no hay zips en {CACHE}"

for anio, archivos in por_anio.items():
    meses = {int(patron.search(z.name)[2]) for z in archivos}
    faltan = sorted(set(range(1, 13)) - meses)
    print(anio, f"{len(archivos):3d} zips | faltan meses:", faltan if faltan else "ninguno")

2020  12 zips | faltan meses: ninguno
2021  12 zips | faltan meses: ninguno
2022  12 zips | faltan meses: ninguno
2023  12 zips | faltan meses: ninguno
2024  12 zips | faltan meses: ninguno
2025  12 zips | faltan meses: ninguno
2026   8 zips | faltan meses: [9, 10, 11, 12]


## 3. Consolidado anual

In [4]:
SALIDA.mkdir(parents=True, exist_ok=True)
ext = ".csv.gz" if COMPRIMIR else ".csv"
BLOQUE = 1 << 20


def abrir(p):
    return gzip.open(p, "wb", compresslevel=6) if COMPRIMIR else open(p, "wb")


for anio, archivos in por_anio.items():
    destino = SALIDA / f"{PAR}-{INTERVALO}-{anio}{ext}"
    if destino.exists() and not SOBRESCRIBIR:
        print("existe, se omite:", destino.name)
        continue
    tmp = destino.with_name(destino.name + ".tmp")
    with abrir(tmp) as out:
        for zp in archivos:
            with zipfile.ZipFile(zp) as z, z.open(z.namelist()[0]) as f:
                ultimo = b"\n"
                primera = f.readline()
                if primera[:1].isdigit():              # no es encabezado: se copia
                    out.write(primera)
                    ultimo = primera[-1:]
                for buf in iter(lambda: f.read(BLOQUE), b""):
                    out.write(buf)
                    ultimo = buf[-1:]
                if ultimo != b"\n":                    # que el archivo siguiente no quede pegado
                    out.write(b"\n")
    tmp.replace(destino)
    print("escrito:", destino.name)

escrito: BTCUSDT-5m-2020.csv
escrito: BTCUSDT-5m-2021.csv
escrito: BTCUSDT-5m-2022.csv
escrito: BTCUSDT-5m-2023.csv
escrito: BTCUSDT-5m-2024.csv
escrito: BTCUSDT-5m-2025.csv
escrito: BTCUSDT-5m-2026.csv


## 4. Resumen

In [5]:
def contar_lineas(p):
    op = gzip.open if p.suffix == ".gz" else open
    with op(p, "rb") as f:
        return sum(buf.count(b"\n") for buf in iter(lambda: f.read(BLOQUE), b""))


for p in sorted(SALIDA.glob(f"{PAR}-{INTERVALO}-*.csv*")):
    if p.suffix == ".tmp":
        continue
    print(f"{p.name:28s} {p.stat().st_size / 2**20:7.1f} MB  {contar_lineas(p):>9,} velas")

BTCUSDT-5m-2020.csv             15.2 MB    105,159 velas
BTCUSDT-5m-2021.csv             15.5 MB    104,923 velas
BTCUSDT-5m-2022.csv             15.6 MB    105,120 velas
BTCUSDT-5m-2023.csv             15.4 MB    105,104 velas
BTCUSDT-5m-2024.csv             15.5 MB    105,408 velas
BTCUSDT-5m-2025.csv             16.2 MB    105,120 velas
BTCUSDT-5m-2026.csv             10.6 MB     69,984 velas
